In [ ]:
%matplotlib widget
import scipp as sc
import plopp as pp
import scippneutron as scn
import scippnexus as snx
import h5py
from pathlib import Path
import numpy as np

In [ ]:
from ess.spectroscopy.indirect import bifrost
from bifrost2409.config import POOCH_DATA_DIR, INTERIM_DATA_DIR
from bifrost2409.dataset import download_datafiles

In [ ]:
datafile = "20240914/BIFROST_20240914T053723.h5"
download_datafiles([datafile])

In [ ]:
targets = [
    'wavelength_monitor',
    'norm_events',
    'triplet_events',
]
target_files = {target: INTERIM_DATA_DIR / f'{Path(datafile).stem}_{target}.h5' for target in targets}
if all(file.exists() for file in target_files.values()):
    from scipp.io import load_hdf5
    from loguru import logger
    from rich.pretty import pretty_repr
    objects = {target: load_hdf5(file) for target, file in target_files.items()}
    logger.info(f'Loaded objects {pretty_repr(target_files)}')
else:
    data = bifrost(POOCH_DATA_DIR / datafile, is_simulated=True)
    objects = {target: data[target] for target in targets}
    for target in targets:
        objects[target].save_hdf5(target_files[target])

In [ ]:
trip = objects['triplet_events']

In [ ]:
events, monitor = [objects[x] for x in ('norm_events', 'wavelength_monitor')]

In [ ]:
from ess.spectroscopy.indirect import bifrost_to_nxspe
nxspe_output = INTERIM_DATA_DIR / 'nxspe' / f'{Path(datafile).stem}'
nxspe_files = bifrost_to_nxspe(events=events, output=nxspe_output)

In [ ]:
# raise ValueError('ok')

In [ ]:
# def hQxz(event_data, x_bins, z_bins, e_range):
#     x = event_data.bins.concat(event_data.dims)
#     x = x.bin(energy_transfer=e_range)[0]
#     x = x.bin(table_momentum_x=x_bins, table_momentum_z=x_bins)
#     return x.hist()

In [ ]:
# hQxz(events, 200, 200, sc.array(values=[-0.01, 0.01], dims=['energy_transfer'], unit='meV')).plot()

In [ ]:
from tqdm import tqdm
#del gr
for i in tqdm(range(monitor.sizes['setting']-2, monitor.sizes['setting'])):
    ev, mn = [x['setting', i] for x in (events, monitor)]
    # gr = ev.bin(incident_wavelength=mn.coords['incident_wavelength'])

In [ ]:
from scipp import sqrt, scalar
from scipp.constants import Planck, neutron_mass

def lambda_to_ei(incident_wavelength):
    return ((Planck / incident_wavelength)**2 / neutron_mass / 2).to(unit='meV')

def ei_ef_to_en(incident_energy, final_energy):
    return incident_energy - final_energy

In [ ]:
targets = ['energy_transfer', 'incident_energy', 'incident_wavelength']
graph = {
        'incident_energy': lambda_to_ei,
        'energy_transfer': ei_ef_to_en,
    }

In [ ]:
def named_coords_midpoint_broadcast(data, names):
    from scipp import midpoints
    sizes = data.sizes
    def coord_midpoint_broadcast(coord):
        for x in coord.dims:
            if x in sizes and coord.sizes[x] == 1 + sizes[x]:
                coord = midpoints(coord, x)
        return coord.broadcast(sizes=sizes)
    
    return {k: coord_midpoint_broadcast(data.coords[k]) for k in names}

def get_empty_centres(data, keep: list[str], graph: dict, extract: list[str], dim: str):
    from scipp import array
    # histogram first
    data = data.hist().transform_coords(keep, graph=graph)
    empty = data.values == 0  # as used below, equivalent to numpy.nonzero(data.values == 0)
    coords = named_coords_midpoint_broadcast(data, extract)
    for k, v in coords.items():
        coords[k] = array(values=v.values[empty], dims=[dim], unit=v.unit, dtype=v.dtype)
        if v.variances is not None:
            coords[k].variances = v.variances[empty]
    return coords, empty


In [ ]:
def initialize_needed(sizes, coord):
    from scipp import DType, full
    dtype = coord.dtype
    if dtype == DType.float64 or dtype == DType.float32:
        from numpy import nan
        default = nan
    elif dtype == DType.int32 or dtype == DType.int64:
        default = -1
    elif dtype == DType.string:
        default = ""
    elif dtype == DType.bool:
        default = False
    elif dtype == DType.datetime64:
        from scipp import datetime
        default = datetime(0)
    else:
        default = -1
    return full(sizes=sizes, unit=coord.unit, dtype=dtype, value=default)


In [ ]:
def clean_up_observations(events, keep: list[str]):
    coords = [x for x in events.bins.coords if x not in keep]
    for coord in coords:
        del events.bins.coords[coord]
    return events

def add_null_observations(tevents, targets: list[str], graph: dict):
    # In the future it may be possible to do this without re-binning at the end
    # https://github.com/scipp/scipp/issues/1967#issuecomment-958680504
    from scipp import DataArray, zeros, concat, min as scipp_min, max as scipp_max
    from loguru import logger

    needed = tevents.dims
    orig = tevents.bins.constituents['data']
    dim = orig.dims[0]
    if any(n not in orig.coords for n in needed):
        bin_begin = tevents.bins.constituents['begin'].values.flatten()
        bin_end = tevents.bins.constituents['end'].values.flatten()
        for n in needed:
            t = initialize_needed(orig.sizes, tevents.coords[n])
            for v, b, e in zip(tevents.coords[n].values.flatten(), bin_begin, bin_end):
                t.values[b:e] = v
            orig.coords[n] = t

    extract = targets + [n for n in needed if n not in targets]
    coords, _ = get_empty_centres(tevents, targets, graph, extract, dim)
    rows = max(coords[target].sizes[dim] for target in targets)
    nulls = DataArray(zeros(sizes={dim: rows}, unit='counts', dtype=orig.dtype), coords=coords)
    if tevents.variances is not None:
        nulls.variances = 1 + nulls.values

    # extend the list of events
    comb = concat((orig, nulls), dim)

    # then bin or combine the coordinates of the input events
    binned = {k: tevents.coords[k] for k in extract if k in tevents.coords and tevents.coords[k].sizes[k] == 1 + tevents.sizes[k]}
    grouped = [k for k in extract if k in tevents.coords and k not in binned]
    # any other extract entries are left on the events' coordinates

    out = comb
    for group in grouped:
        smallest = scipp_min(tevents.coords[group])
        largest = scipp_max(tevents.coords[group])
        largest.value += 1  # label based slicing is upper-bound exclusive :(
        out = out.group(group)[group, smallest:largest]
    out = out.bin(binned)
    for k, v in tevents.coords.items():
        out.coords[k] = v
    return out

In [ ]:
## max_int = sc.concat([sc.max(add_null_observations(clean_up_observations(events['setting', x], targets), targets, graph).hist()) for x in range(events.sizes['setting'])], 'setting')
#max_int = sc.concat([sc.max(clean_up_observations(events['setting', x], targets).hist()) for x in range(events.sizes['setting'])], 'setting')

In [ ]:
#max_int.rename_dims(setting='a3').plot()

In [ ]:
from scipp import DataArray
def _add_null_observations_append(events, targets: list[str], graph: dict):
    """Extend the events list after finding the number of empty bins. Then update the
    bin-indexing for the empty bins to point at the added end-of-list null events

    If done correctly, the input event information is undisturbed and the null events
    get assigned to bins which were otherwise unused.
    """
    from scipp import zeros, concat
    input_event_list = events.bins.constituents['data']
    needed = events.dims
    dim = input_event_list.dims[0]
    extract = targets + [n for n in needed if n not in targets]
    coords, empty = get_empty_centres(events, targets, graph, extract, dim)
    rows = max(coords[target].sizes[dim] for target in targets)
    nulls = DataArray(
        zeros(sizes={dim: rows}, unit='counts', dtype=input_event_list.dtype),
        coords=coords
    )
    if events.variances is not None:
        nulls.variances = 1 + nulls.values
    # make the new event list by concatenating the null observations on the end
    output_event_list = concat((input_event_list, nulls), dim=dim)
    # the first index for the _newly added_ null observations in the event list
    first = input_event_list.sizes[dim]
    return first, empty, nulls.sizes[dim]



In [ ]:
cev = clean_up_observations(ev, targets)
fst, mt, nonull = _add_null_observations_append(cev, targets, graph)

In [ ]:
np.cumsum(mt)

In [ ]:
[x.shape for x in np.nonzero(mt)]

In [ ]:
nonull

In [ ]:
averages = {t: cev.bins.coords[t].bins.nanmean() for t in targets}

In [ ]:
list(averages)

In [ ]:
en = averages['energy_transfer']
en.plot()

In [ ]:
hev = cev.hist().transform_coords(targets, graph=graph)
hev.coords['energy_transfer'].plot()

In [ ]:
hev.coords['energy_transfer'].mean()